Explore CLI prompts, workbook synchronization and run-default precedence. Examples use local values; no Databricks connection is made.


In [ ]:
from setup.nbs.inspect_tools import show, show_methods
from pair_recon.app.cli import PromptSession, prompts, persistent_settings
from pair_recon.app.cli import synchronize_pair_workbook, synchronize_key_types
from pair_recon.app.cli import select_last_failed_run, add_mismatched_partitions

show_methods(PromptSession, 'begin', 'answer', 'select', 'select_many', 'confirm', 'worker_budget')
show(prompts, persistent_settings, synchronize_pair_workbook, synchronize_key_types)
show(select_last_failed_run, add_mismatched_partitions)


## Resolve saved and catalog defaults

Saved values normally take precedence. When catalog metadata is authoritative, catalog-owned values and run defaults take precedence, including an absent resettable `use_modular`.


In [ ]:
from pprint import pprint
from pair_recon.app.config import resolve_run_defaults

metadata = {"partition": "month", "_run_defaults": {"sample": 0.1}}
saved = {"partition": "day", "sample": 0.5, "use_modular": True}
print("metadata:"); pprint(metadata)
print("saved:"); pprint(saved)
print("workbook precedence:"); pprint(resolve_run_defaults(metadata, saved))
print("catalog precedence:"); pprint(resolve_run_defaults(metadata, saved, catalog_authoritative=True))


## Workbook synchronization skips disabled pair mappings

A temporary file and mocks show which rows are synchronized; the example does not connect to a database.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from unittest.mock import Mock, patch

rows = [{"pair_name": "ON", "enabled": True}, {"pair_name": "OFF", "enabled": False}]
catalog = Mock()
catalog.configuration_rows.return_value = []
with TemporaryDirectory() as directory:
    path = Path(directory) / "pairs.xlsx"
    path.touch()
    with patch("pair_recon.app.cli.read_pair_workbook", return_value=rows), \
         patch("pair_recon.app.cli.changed_pair_configurations", return_value=[]), \
         patch("pair_recon.app.cli.synchronize_key_types", return_value=1) as sync:
        loaded, result = synchronize_pair_workbook(Mock(), catalog, "demo.recon.pair_recon_pair_meta", path)
        print("input pairs:", [row["pair_name"] for row in loaded])
        print("mapping rows:", result["mapping_rows"])
        print("synchronized pairs:", [call.args[2]["pair_name"] for call in sync.call_args_list])


## `PromptSession` input methods

Mock the terminal helpers so the example is deterministic. The recorded answers demonstrate `begin`, `answer`, `select`, `select_many`, `confirm`, and `worker_budget`.


In [ ]:
from unittest.mock import patch

session = PromptSession()
session.begin()
with patch("pair_recon.app.cli.ask", return_value="alex") as ask_mock, \
     patch("pair_recon.app.cli.choose", return_value="orders") as choose_mock, \
     patch("pair_recon.app.cli.choose_many", return_value=["min", "max"]) as many_mock, \
     patch("pair_recon.app.cli.yes_no", return_value=True) as confirm_mock, \
     patch("pair_recon.app.cli.positive_int", return_value=4) as workers_mock:
    print("answer:", session.answer("Owner", "guest"), "input:", ask_mock.call_args)
    print("select:", session.select("Pair", ["orders", "items"], "items"), "input:", choose_mock.call_args)
    print("select_many:", session.select_many("Checks", ["min", "max"], ["min"]), "input:", many_mock.call_args)
    print("confirm:", session.confirm("Continue", False), "input:", confirm_mock.call_args)
    print("worker_budget:", session.worker_budget("Workers", 2), "input:", workers_mock.call_args)
print("stored answers:", session.answers)
print("cursor before begin:", session.cursor)
session.begin()
print("cursor after begin:", session.cursor)


## Persist settings and select the latest failed run

Transient command flags are removed from saved settings. A mock catalog supplies the previous run.


In [ ]:
from unittest.mock import Mock
settings = {"pair": "orders", "owner": "alex", "run_id": "old", "force": True, "_internal": 1}
print("input settings:", settings)
print("persistent settings:", persistent_settings(settings))
catalog = Mock()
catalog.latest_run.return_value = {"run_id": "orders-alex-20261109", "state": "failed"}
print("latest run:", catalog.latest_run.return_value)
print("action:", select_last_failed_run(settings, catalog))
print("selected run:", settings["run_id"])
print("catalog input:", catalog.latest_run.call_args)


## Synchronize key types without a database

The mock catalog provides existing mappings; the mocked upsert exposes the exact rows and overwrite mode that would be sent.


In [ ]:
from unittest.mock import Mock, patch
catalog = Mock()
catalog.column_map_table = "demo.recon.pair_recon_column_map"
catalog.column_mappings.return_value = []
row = {"pair_name": "orders", "key_columns": ["ORDER_ID[numeric]"], "left_date_col": "ORDER_DT", "right_date_col": "order_date"}
with patch("pair_recon.app.cli.upsert_column_mappings") as upsert:
    count = synchronize_key_types(Mock(), catalog, row)
    print("input:", row)
    print("changed mappings:", count)
    print("upsert target:", upsert.call_args.args[1])
    print("upsert pair:", upsert.call_args.args[2])
    print("upsert mappings:", upsert.call_args.args[3])
    print("upsert options:", upsert.call_args.kwargs)


## Add mismatched partitions

Mock the error parser to make the partition input explicit; the real function updates both settings and pair configuration.


In [ ]:
settings = {"pair": "orders", "excluded_partitions": ["2026-01-01"]}
config = {"pairs": {"orders": {}}}
with patch("pair_recon.app.cli.mismatched_partitions", return_value=["2026-01-02"]):
    print("input settings:", settings)
    print("added:", add_mismatched_partitions(settings, config, RuntimeError("toy row mismatch")))
    print("updated settings:", settings)
    print("updated configuration:", config)
